In [7]:
import os
import sys
import subprocess
import re
import pandas as pd
import json
import glob
from datetime import datetime, timedelta
from collections import defaultdict
import hashlib

# ======================
# CONFIGURACIÓN INICIAL
# ======================
CONFIG = {
    'ruta_carpeta': r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\INGRESAR DE FM\IMPUT-AGREGA-ARCHIVOS",
    'patrones_aprendidos': 'patrones_aprendidos.json',
    'modelo_ia': 'modelo_extraccion.json',
    'max_iteraciones': 100,
    'umbral_confianza': 0.8
}

# ======================
# SISTEMA DE APRENDIZAJE
# ======================
class SistemaAprendizaje:
    def __init__(self):
        self.patrones = self.cargar_patrones()
        self.modelo = self.cargar_modelo()
        self.resultados_validos = []
        
    def cargar_patrones(self):
        """Carga patrones aprendidos de ejecuciones anteriores"""
        try:
            with open(CONFIG['patrones_aprendidos'], 'r', encoding='utf-8') as f:
                return json.load(f)
        except:
            # Patrones iniciales basados en conocimiento común
            return {
                'fecha_hora': [
                    r'(\d{2}/\d{2}/\d{4}\s+\d{1,2}:\d{2})',
                    r'(\d{2}-\d{2}-\d{4}\s+\d{1,2}:\d{2})',
                    r'Día:\s*(\d{2}/\d{2}/\d{4})',
                    r'Fecha:\s*(\d{2}/\d{2}/\d{4})',
                    r'Hora:\s*(\d{1,2}:\d{2})'
                ],
                'alimentador': [
                    r'Alimentador:\s*([A-Z]{1,3}[-_][A-Z0-9]{1,3}(?:\s*,\s*[A-Z]{1,3}[-_][A-Z0-9]{1,3})*)',
                    r'alimentador\s+([A-Z]{1,3}[-_][A-Z0-9]{1,3})',
                    r'([A-Z]{1,3}[-_][A-Z0-9]{1,3})\s*\(alimentador\)',
                    r'Instalación:\s*([^\n]+)',
                    r'([A-Z]{2,3}-\d{2,3})'
                ],
                'reposicion': [
                    r'Reposición:\s*(\d{2}/\d{2}/\d{4}\s+\d{1,2}:\d{2})',
                    r'Normalización:\s*(\d{2}/\d{2}/\d{4}\s+\d{1,2}:\d{2})',
                    r'Restablecimiento:\s*(\d{2}/\d{2}/\d{4}\s+\d{1,2}:\d{2})',
                    r'Hora de normalización:\s*(\d{1,2}:\d{2})'
                ],
                'lugares': [
                    r'Lugares afectados:\s*([^\n]+(?:\n[^\n]+){0,3})',
                    r'Sectores:\s*([^\n]+(?:\n[^\n]+){0,3})',
                    r'Clientes afectados:\s*([^\n]+(?:\n[^\n]+){0,3})',
                    r'Ubicación:\s*([^\n]+(?:\n[^\n]+){0,3})',
                    r'Área afectada:\s*([^\n]+(?:\n[^\n]+){0,3})'
                ]
            }
    
    def cargar_modelo(self):
        """Carga el modelo de IA entrenado"""
        try:
            with open(CONFIG['modelo_ia'], 'r', encoding='utf-8') as f:
                return json.load(f)
        except:
            return {
                'patrones_exitosos': defaultdict(int),
                'patrones_fallidos': defaultdict(int),
                'estructuras_detectadas': [],
                'ejemplos_validos': []
            }
    
    def guardar_aprendizaje(self):
        """Guarda lo aprendido para futuras ejecuciones"""
        with open(CONFIG['patrones_aprendidos'], 'w', encoding='utf-8') as f:
            json.dump(self.patrones, f, indent=2, ensure_ascii=False)
        
        with open(CONFIG['modelo_ia'], 'w', encoding='utf-8') as f:
            json.dump(self.modelo, f, indent=2, ensure_ascii=False)
    
    def agregar_patron_exitoso(self, categoria, patron, contexto):
        """Registra un patrón que funcionó bien"""
        clave = f"{categoria}:{patron}"
        self.modelo['patrones_exitosos'][clave] += 1
        
        # Si es muy exitoso, agregarlo a patrones principales
        if self.modelo['patrones_exitosos'][clave] > 3:
            if patron not in self.patrones.get(categoria, []):
                self.patrones.setdefault(categoria, []).insert(0, patron)
    
    def analizar_estructura(self, texto, datos_extraidos):
        """Analiza la estructura del documento para mejorar extracción"""
        lineas = texto.split('\n')
        estructura = []
        
        for i, linea in enumerate(lineas):
            linea_limpia = linea.strip()
            if len(linea_limpia) > 10:  # Líneas significativas
                estructura.append({
                    'linea': i,
                    'texto': linea_limpia[:50],
                    'tiene_fecha': bool(re.search(r'\d{2}/\d{2}/\d{4}', linea_limpia)),
                    'tiene_hora': bool(re.search(r'\d{1,2}:\d{2}', linea_limpia)),
                    'tiene_alimentador': bool(re.search(r'[A-Z]{1,3}[-_][A-Z0-9]{1,3}', linea_limpia, re.I))
                })
        
        self.modelo['estructuras_detectadas'].append({
            'hash': hashlib.md5(texto.encode()).hexdigest()[:10],
            'estructura': estructura[:10],  # Solo primeras 10 líneas
            'datos_extraidos': datos_extraidos
        })
    
    def sugerir_patrones(self, texto, datos_faltantes):
        """Sugiere nuevos patrones basados en el contenido"""
        sugerencias = {}
        
        if 'fecha_inicio' in datos_faltantes:
            # Buscar patrones de fecha no utilizados
            fechas = re.findall(r'\b\d{2}[/\-]\d{2}[/\-]\d{4}\b', texto)
            for fecha in set(fechas):
                patron = re.escape(fecha)
                if patron not in str(self.patrones['fecha_hora']):
                    sugerencias.setdefault('fecha_hora', []).append(patron)
        
        if 'alimentador' in datos_faltantes:
            # Buscar posibles códigos de alimentador
            posibles = re.findall(r'\b[A-Z]{1,3}[-_ ][A-Z0-9]{1,3}\b', texto)
            for posible in set(posibles):
                patron = re.escape(posible)
                if patron not in str(self.patrones['alimentador']):
                    sugerencias.setdefault('alimentador', []).append(patron)
        
        return sugerencias

# ======================
# MOTOR DE EXTRACCIÓN MULTICAPA
# ======================
class MotorExtraccion:
    def __init__(self, sistema_aprendizaje):
        self.sa = sistema_aprendizaje
        
    def extraer_texto_word(self, ruta_archivo):
        """Extrae texto de archivos Word usando múltiples métodos"""
        metodos = [
            self._metodo_python_docx,
            self._metodo_zip_xml,
            self._metodo_texto_plano,
            self._metodo_win32com
        ]
        
        for metodo in metodos:
            try:
                texto = metodo(ruta_archivo)
                if texto and len(texto) > 100:  # Texto significativo
                    return texto
            except:
                continue
        
        return None
    
    def _metodo_python_docx(self, ruta_archivo):
        """Método 1: Usar python-docx"""
        from docx import Document
        doc = Document(ruta_archivo)
        return "\n".join([para.text for para in doc.paragraphs])
    
    def _metodo_zip_xml(self, ruta_archivo):
        """Método 2: Usar ZIP y XML (para .docx)"""
        import zipfile
        import xml.etree.ElementTree as ET
        
        with zipfile.ZipFile(ruta_archivo, 'r') as docx:
            xml_content = docx.read('word/document.xml').decode('utf-8', errors='ignore')
        
        # Extraer texto entre tags <w:t>
        textos = re.findall(r'<w:t[^>]*>([^<]+)</w:t>', xml_content)
        return " ".join(textos)
    
    def _metodo_texto_plano(self, ruta_archivo):
        """Método 3: Leer como texto plano"""
        with open(ruta_archivo, 'rb') as f:
            contenido = f.read()
        
        # Intentar diferentes encodings
        encodings = ['utf-8', 'latin-1', 'cp1252']
        for encoding in encodings:
            try:
                return contenido.decode(encoding, errors='ignore')
            except:
                continue
        return None
    
    def _metodo_win32com(self, ruta_archivo):
        """Método 4: Usar win32com (solo Windows)"""
        if sys.platform == 'win32':
            import win32com.client
            word = win32com.client.Dispatch("Word.Application")
            word.Visible = False
            doc = word.Documents.Open(os.path.abspath(ruta_archivo))
            texto = doc.Content.Text
            doc.Close()
            word.Quit()
            return texto
        return None
    
    def extraer_datos_avanzado(self, texto, nombre_archivo):
        """Extracción multicapa con validación cruzada"""
        datos = {
            'fecha_inicio': None,
            'alimentador': None,
            'fecha_reposicion': None,
            'lugares': None,
            'confianza': 0.0,
            'metodos_usados': []
        }
        
        # CAPA 1: Patrones predefinidos
        capa1 = self._capa_patrones(texto)
        datos.update(capa1['datos'])
        datos['metodos_usados'].append('patrones')
        
        # CAPA 2: Análisis de posición y contexto
        if datos['confianza'] < CONFIG['umbral_confianza']:
            capa2 = self._capa_contexto(texto, datos)
            if capa2['confianza'] > datos['confianza']:
                datos.update(capa2['datos'])
                datos['confianza'] = capa2['confianza']
            datos['metodos_usados'].append('contexto')
        
        # CAPA 3: Aprendizaje automático
        if datos['confianza'] < CONFIG['umbral_confianza']:
            capa3 = self._capa_aprendizaje(texto, nombre_archivo)
            if capa3['confianza'] > datos['confianza']:
                datos.update(capa3['datos'])
                datos['confianza'] = capa3['confianza']
            datos['metodos_usados'].append('ia')
        
        # CAPA 4: Inferencia lógica
        if datos['confianza'] < CONFIG['umbral_confianza']:
            capa4 = self._capa_inferencia(texto, datos)
            if capa4['confianza'] > datos['confianza']:
                datos.update(capa4['datos'])
                datos['confianza'] = capa4['confianza']
            datos['metodos_usados'].append('inferencia')
        
        # Validar y limpiar datos
        datos = self._validar_datos(datos)
        
        return datos
    
    def _capa_patrones(self, texto):
        """Capa 1: Extracción por patrones de regex"""
        resultados = {
            'fecha_inicio': None,
            'alimentador': None,
            'fecha_reposicion': None,
            'lugares': None,
            'confianza': 0.0
        }
        
        puntuacion = 0
        max_puntos = 4
        
        # Extraer fecha inicio
        for patron in self.sa.patrones['fecha_hora']:
            match = re.search(patron, texto)
            if match:
                resultados['fecha_inicio'] = match.group(1)
                self.sa.agregar_patron_exitoso('fecha_hora', patron, match.group(0))
                puntuacion += 1
                break
        
        # Extraer alimentador
        for patron in self.sa.patrones['alimentador']:
            match = re.search(patron, texto, re.IGNORECASE)
            if match:
                resultados['alimentador'] = match.group(1).strip()
                self.sa.agregar_patron_exitoso('alimentador', patron, match.group(0))
                puntuacion += 1
                break
        
        # Extraer reposición
        for patron in self.sa.patrones['reposicion']:
            match = re.search(patron, texto, re.IGNORECASE)
            if match:
                resultados['fecha_reposicion'] = match.group(1).strip()
                self.sa.agregar_patron_exitoso('reposicion', patron, match.group(0))
                puntuacion += 1
                break
        
        # Extraer lugares
        for patron in self.sa.patrones['lugares']:
            match = re.search(patron, texto, re.IGNORECASE | re.DOTALL)
            if match:
                lugares = match.group(1).strip()
                # Limpiar el texto
                lugares = re.sub(r'\s+', ' ', lugares)
                lugares = re.sub(r'\n+', ', ', lugares)
                resultados['lugares'] = lugares[:500]  # Limitar longitud
                self.sa.agregar_patron_exitoso('lugares', patron, match.group(0)[:100])
                puntuacion += 1
                break
        
        resultados['confianza'] = puntuacion / max_puntos
        return {'datos': resultados, 'confianza': resultados['confianza']}
    
    def _capa_contexto(self, texto, datos_actuales):
        """Capa 2: Análisis contextual y por posición"""
        lineas = texto.split('\n')
        resultados = datos_actuales.copy()
        confianza = datos_actuales['confianza']
        
        # Buscar patrones contextuales
        for i, linea in enumerate(lineas):
            linea_lower = linea.lower()
            
            # Buscar "inicio" cerca de fechas
            if 'inicio' in linea_lower and not resultados['fecha_inicio']:
                # Buscar fecha en esta línea o cercanas
                for j in range(max(0, i-2), min(len(lineas), i+3)):
                    fechas = re.findall(r'\d{2}/\d{2}/\d{4}\s+\d{1,2}:\d{2}', lineas[j])
                    if fechas:
                        resultados['fecha_inicio'] = fechas[0]
                        confianza = max(confianza, 0.7)
                        break
            
            # Buscar "alimentador" o "instalación"
            if any(term in linea_lower for term in ['alimentador', 'instalación', 'circuito']):
                # Buscar código de alimentador
                codigos = re.findall(r'[A-Z]{1,3}[-_][A-Z0-9]{1,3}', linea, re.I)
                if codigos and not resultados['alimentador']:
                    resultados['alimentador'] = ', '.join(set(codigos))
                    confianza = max(confianza, 0.6)
            
            # Buscar "afectados" para lugares
            if any(term in linea_lower for term in ['afectados', 'sectores', 'ubicación']):
                # Tomar esta línea y las siguientes 3 como lugares
                lugares_texto = ' '.join(lineas[i:i+4])
                # Limpiar
                lugares_texto = re.sub(r'^.*?[:\-]\s*', '', lugares_texto, flags=re.I)
                lugares_texto = re.sub(r'\s+', ' ', lugares_texto).strip()
                if len(lugares_texto) > 10 and not resultados['lugares']:
                    resultados['lugares'] = lugares_texto[:500]
                    confianza = max(confianza, 0.5)
        
        return {'datos': resultados, 'confianza': confianza}
    
    def _capa_aprendizaje(self, texto, nombre_archivo):
        """Capa 3: Aprendizaje automático basado en ejemplos"""
        resultados = {
            'fecha_inicio': None,
            'alimentador': None,
            'fecha_reposicion': None,
            'lugares': None,
            'confianza': 0.0
        }
        
        # Buscar en ejemplos anteriores similares
        hash_texto = hashlib.md5(texto.encode()).hexdigest()[:10]
        
        # Verificar si tenemos estructuras similares
        for estructura in self.sa.modelo['estructuras_detectadas']:
            # Similitud básica por longitud y palabras clave
            palabras_clave = ['interrupción', 'alimentador', 'afectados', 'reposición']
            coincidencias = sum(1 for palabra in palabras_clave if palabra in texto.lower())
            
            if coincidencias >= 2 and 'datos_extraidos' in estructura:
                # Usar datos de estructura similar
                datos_similares = estructura['datos_extraidos']
                if datos_similares.get('fecha_inicio'):
                    resultados['fecha_inicio'] = datos_similares.get('fecha_inicio')
                if datos_similares.get('alimentador'):
                    resultados['alimentador'] = datos_similares.get('alimentador')
                if datos_similares.get('fecha_reposicion'):
                    resultados['fecha_reposicion'] = datos_similares.get('fecha_reposicion')
                if datos_similares.get('lugares'):
                    resultados['lugares'] = datos_similares.get('lugares')
                
                resultados['confianza'] = 0.6
                break
        
        return {'datos': resultados, 'confianza': resultados['confianza']}
    
    def _capa_inferencia(self, texto, datos_actuales):
        """Capa 4: Inferencia lógica y heurística"""
        resultados = datos_actuales.copy()
        confianza = datos_actuales['confianza']
        
        # Inferir fecha de inicio si hay fecha pero no hora
        if resultados['fecha_inicio'] and ':' not in resultados['fecha_inicio']:
            # Buscar hora cercana
            hora_match = re.search(r'(\d{1,2}:\d{2})', texto)
            if hora_match:
                resultados['fecha_inicio'] = f"{resultados['fecha_inicio']} {hora_match.group(1)}"
                confianza = max(confianza, 0.4)
        
        # Inferir alimentador del nombre del archivo
        if not resultados['alimentador']:
            # Buscar patrones en el nombre del archivo
            archivo_match = re.search(r'([A-Z]{1,3}[-_][A-Z0-9]{1,3})', texto.split('\n')[0] if '\n' in texto else texto)
            if archivo_match:
                resultados['alimentador'] = archivo_match.group(1)
                confianza = max(confianza, 0.3)
        
        # Inferir reposición como inicio + X horas
        if resultados['fecha_inicio'] and not resultados['fecha_reposicion']:
            try:
                # Intentar parsear fecha de inicio
                fecha_obj = datetime.strptime(resultados['fecha_inicio'], '%d/%m/%Y %H:%M')
                # Asumir 1-4 horas después (heurística)
                fecha_repos = fecha_obj + timedelta(hours=2)
                resultados['fecha_reposicion'] = fecha_repos.strftime('%d/%m/%Y %H:%M')
                confianza = max(confianza, 0.2)
            except:
                pass
        
        # Extraer lugares del texto general si no se encontraron
        if not resultados['lugares']:
            # Buscar líneas que contengan palabras relacionadas con lugares
            lineas = texto.split('\n')
            lugares_candidatos = []
            for linea in lineas:
                if any(palabra in linea.lower() for palabra in ['calle', 'av.', 'jr.', 'urb.', 'sector', 'asoc.', 'fundo']):
                    lugares_candidatos.append(linea.strip())
            
            if lugares_candidatos:
                resultados['lugares'] = '; '.join(lugares_candidatos[:5])  # Máximo 5
                confianza = max(confianza, 0.3)
        
        resultados['confianza'] = confianza
        return {'datos': resultados, 'confianza': confianza}
    
    def _validar_datos(self, datos):
        """Valida y limpia los datos extraídos"""
        # Validar formato de fecha
        for campo in ['fecha_inicio', 'fecha_reposicion']:
            if datos[campo]:
                # Intentar estandarizar formato
                try:
                    # Probar diferentes formatos
                    formatos = ['%d/%m/%Y %H:%M', '%d/%m/%Y', '%d-%m-%Y %H:%M']
                    fecha_obj = None
                    for fmt in formatos:
                        try:
                            fecha_obj = datetime.strptime(datos[campo], fmt)
                            break
                        except:
                            continue
                    
                    if fecha_obj:
                        datos[campo] = fecha_obj.strftime('%d/%m/%Y %H:%M')
                except:
                    pass
        
        # Validar alimentador (limpiar caracteres extraños)
        if datos['alimentador']:
            datos['alimentador'] = re.sub(r'[^\w\s\-,]', '', datos['alimentador']).strip()
        
        # Validar lugares (limitar longitud)
        if datos['lugares'] and len(datos['lugares']) > 500:
            datos['lugares'] = datos['lugares'][:497] + '...'
        
        return datos

# ======================
# SISTEMA DE VALIDACIÓN
# ======================
class SistemaValidacion:
    def __init__(self):
        self.reglas = self._cargar_reglas()
    
    def _cargar_reglas(self):
        """Carga reglas de validación"""
        return {
            'fecha_formato': r'\d{2}/\d{2}/\d{4}(?:\s+\d{1,2}:\d{2})?',
            'alimentador_formato': r'^[A-Z]{1,3}[-_][A-Z0-9]{1,3}(?:\s*,\s*[A-Z]{1,3}[-_][A-Z0-9]{1,3})*$',
            'lugares_min_length': 10
        }
    
    def validar_registro(self, registro):
        """Valida un registro individual"""
        errores = []
        
        # Validar fecha de inicio
        if not registro.get('fecha_inicio'):
            errores.append("Falta fecha de inicio")
        elif not re.match(self.reglas['fecha_formato'], registro['fecha_inicio']):
            errores.append(f"Formato fecha inválido: {registro['fecha_inicio']}")
        
        # Validar alimentador
        if not registro.get('alimentador'):
            errores.append("Falta alimentador")
        elif not re.match(self.reglas['alimentador_formato'], registro['alimentador'], re.IGNORECASE):
            errores.append(f"Formato alimentador inválido: {registro['alimentador']}")
        
        # Validar lugares
        if not registro.get('lugares'):
            errores.append("Falta información de lugares")
        elif len(registro['lugares']) < self.reglas['lugares_min_length']:
            errores.append(f"Información de lugares muy corta: {registro['lugares'][:50]}...")
        
        return {
            'valido': len(errores) == 0,
            'errores': errores,
            'confianza': registro.get('confianza', 0)
        }

# ======================
# INTERFAZ ITERATIVA
# ======================
class InterfazIterativa:
    def __init__(self, motor, validacion, sistema_aprendizaje):
        self.motor = motor
        self.validacion = validacion
        self.sa = sistema_aprendizaje
        self.resultados = []
        self.estadisticas = {
            'total_archivos': 0,
            'exitosos': 0,
            'parciales': 0,
            'fallidos': 0,
            'confianza_promedio': 0.0
        }
    
    def procesar_carpeta(self, ruta_carpeta):
        """Procesa todos los archivos Word en la carpeta"""
        # Buscar archivos
        extensiones = ['*.docx', '*.doc', '*.DOCX', '*.DOC', '*.rtf', '*.txt']
        archivos = []
        for ext in extensiones:
            archivos.extend(glob.glob(os.path.join(ruta_carpeta, ext)))
        
        self.estadisticas['total_archivos'] = len(archivos)
        
        if not archivos:
            print("❌ No se encontraron archivos para procesar")
            return []
        
        print(f"📂 Encontrados {len(archivos)} archivos")
        print("="*80)
        
        # Procesar cada archivo
        for i, archivo in enumerate(archivos, 1):
            print(f"\n[{i}/{len(archivos)}] Procesando: {os.path.basename(archivo)}")
            print("-"*80)
            
            # Extraer texto
            texto = self.motor.extraer_texto_word(archivo)
            if not texto:
                print("  ❌ No se pudo extraer texto del archivo")
                self.estadisticas['fallidos'] += 1
                continue
            
            print(f"  📝 Texto extraído ({len(texto)} caracteres)")
            
            # Extraer datos
            datos = self.motor.extraer_datos_avanzado(texto, os.path.basename(archivo))
            
            # Validar
            validacion = self.validacion.validar_registro(datos)
            
            # Mostrar resultados
            self._mostrar_resultado(datos, validacion)
            
            # Analizar estructura para aprendizaje
            self.sa.analizar_estructura(texto[:1000], datos)
            
            # Guardar resultados
            registro = {
                'archivo': os.path.basename(archivo),
                'fecha_inicio': datos['fecha_inicio'],
                'alimentador': datos['alimentador'],
                'fecha_reposicion': datos['fecha_reposicion'],
                'lugares': datos['lugares'],
                'confianza': datos['confianza'],
                'metodos': datos['metodos_usados'],
                'valido': validacion['valido'],
                'errores': validacion['errores']
            }
            
            self.resultados.append(registro)
            
            # Actualizar estadísticas
            if validacion['valido'] and datos['confianza'] >= 0.8:
                self.estadisticas['exitosos'] += 1
            elif datos['confianza'] >= 0.4:
                self.estadisticas['parciales'] += 1
            else:
                self.estadisticas['fallidos'] += 1
            
            self.estadisticas['confianza_promedio'] = (
                (self.estadisticas['confianza_promedio'] * (i-1) + datos['confianza']) / i
            )
        
        return self.resultados
    
    def _mostrar_resultado(self, datos, validacion):
        """Muestra los resultados de la extracción"""
        print("  📊 Resultados de extracción:")
        print(f"    • Fecha inicio: {datos['fecha_inicio'] or 'NO ENCONTRADO'}")
        print(f"    • Alimentador: {datos['alimentador'] or 'NO ENCONTRADO'}")
        print(f"    • Fecha reposición: {datos['fecha_reposicion'] or 'NO ENCONTRADO'}")
        print(f"    • Lugares: {datos['lugares'][:100] + '...' if datos['lugares'] and len(datos['lugares']) > 100 else datos['lugares'] or 'NO ENCONTRADO'}")
        print(f"    • Confianza: {datos['confianza']:.2%}")
        print(f"    • Métodos usados: {', '.join(datos['metodos_usados'])}")
        
        if validacion['valido']:
            print("  ✅ Validación: APROBADO")
        else:
            print(f"  ⚠️  Validación: RECHAZADO - {', '.join(validacion['errores'])}")
    
    def generar_excel(self, ruta_salida):
        """Genera el archivo Excel final"""
        if not self.resultados:
            print("❌ No hay datos para generar Excel")
            return None
        
        # Filtrar registros válidos o con buena confianza
        registros_validos = [r for r in self.resultados if r['valido'] or r['confianza'] >= 0.5]
        
        if not registros_validos:
            print("⚠️  No hay registros suficientemente buenos para Excel")
            return None
        
        # Crear DataFrame con formato específico
        datos_excel = []
        for registro in registros_validos:
            datos_excel.append({
                'Fecha y Hora de Inicio de Interrupción': registro['fecha_inicio'] or 'NO DISPONIBLE',
                'Instalación Afectada': registro['alimentador'] or 'NO DISPONIBLE',
                'Fecha y Hora de Reposición': registro['fecha_reposicion'] or 'NO DISPONIBLE',
                'Lugares y Sectores Afectados': registro['lugares'] or 'NO DISPONIBLE'
            })
        
        df = pd.DataFrame(datos_excel)
        
        # Generar nombre único con timestamp
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        archivo_excel = os.path.join(ruta_salida, f"AVISOS_INTERRUPCION_{timestamp}.xlsx")
        
        try:
            # Guardar Excel
            df.to_excel(archivo_excel, index=False)
            
            # También guardar CSV
            archivo_csv = archivo_excel.replace('.xlsx', '.csv')
            df.to_csv(archivo_csv, index=False, sep=';', encoding='utf-8-sig')
            
            print(f"\n✅ Excel generado: {archivo_excel}")
            print(f"📄 CSV generado: {archivo_csv}")
            
            # Mostrar vista previa
            print("\n📋 Vista previa del Excel:")
            print("="*100)
            print(df.to_string(index=False))
            print("="*100)
            
            return archivo_excel
            
        except Exception as e:
            print(f"❌ Error al generar Excel: {e}")
            return None
    
    def mostrar_estadisticas(self):
        """Muestra estadísticas del procesamiento"""
        print("\n" + "="*80)
        print("📈 ESTADÍSTICAS FINALES DEL PROCESAMIENTO")
        print("="*80)
        print(f"• Total archivos procesados: {self.estadisticas['total_archivos']}")
        print(f"• Extracciones exitosas (confianza ≥80%): {self.estadisticas['exitosos']}")
        print(f"• Extracciones parciales (40% ≤ confianza < 80%): {self.estadisticas['parciales']}")
        print(f"• Extracciones fallidas (confianza < 40%): {self.estadisticas['fallidos']}")
        print(f"• Confianza promedio: {self.estadisticas['confianza_promedio']:.2%}")
        
        if self.estadisticas['total_archivos'] > 0:
            tasa_exito = (self.estadisticas['exitosos'] / self.estadisticas['total_archivos']) * 100
            print(f"• Tasa de éxito: {tasa_exito:.1f}%")
        
        # Mostrar métodos más usados
        if self.resultados:
            metodos = {}
            for r in self.resultados:
                for metodo in r.get('metodos', []):
                    metodos[metodo] = metodos.get(metodo, 0) + 1
            
            print("\n🔧 Métodos de extracción más usados:")
            for metodo, count in sorted(metodos.items(), key=lambda x: x[1], reverse=True):
                print(f"  • {metodo}: {count} veces")

# ======================
# PROGRAMA PRINCIPAL
# ======================
def main():
    """Función principal del sistema"""
    print("="*80)
    print("🤖 SISTEMA AUTOMÁTICO DE EXTRACCIÓN DE AVISOS DE INTERRUPCIÓN")
    print("="*80)
    
    # Inicializar componentes
    print("🔧 Inicializando componentes...")
    sistema_aprendizaje = SistemaAprendizaje()
    motor = MotorExtraccion(sistema_aprendizaje)
    validacion = SistemaValidacion()
    interfaz = InterfazIterativa(motor, validacion, sistema_aprendizaje)
    
    # Verificar carpeta
    ruta_carpeta = CONFIG['ruta_carpeta']
    if not os.path.exists(ruta_carpeta):
        print(f"❌ La carpeta no existe: {ruta_carpeta}")
        print("Creando carpeta...")
        os.makedirs(ruta_carpeta, exist_ok=True)
        print(f"✅ Carpeta creada. Por favor, coloca los archivos Word en: {ruta_carpeta}")
        return
    
    # Procesar archivos
    print(f"\n🔍 Buscando archivos en: {ruta_carpeta}")
    resultados = interfaz.procesar_carpeta(ruta_carpeta)
    
    # Generar Excel
    if resultados:
        print("\n💾 Guardando resultados...")
        excel_generado = interfaz.generar_excel(ruta_carpeta)
        
        # Guardar aprendizaje
        sistema_aprendizaje.guardar_aprendizaje()
        print("🧠 Aprendizaje guardado para futuras ejecuciones")
    else:
        print("\n⚠️  No se procesaron archivos válidos")
    
    # Mostrar estadísticas
    interfaz.mostrar_estadisticas()
    
    # Generar reporte de problemas
    problemas = [r for r in resultados if not r['valido'] and r['confianza'] < 0.4]
    if problemas:
        print("\n⚠️  ARCHIVOS CON PROBLEMAS (requieren revisión manual):")
        for p in problemas:
            print(f"  • {p['archivo']}: {', '.join(p['errores'])}")
    
    print("\n" + "="*80)
    print("🎉 PROCESO COMPLETADO")
    print("="*80)

# ======================
# EJECUCIÓN
# ======================
if __name__ == "__main__":
    # Verificar e instalar dependencias
    print("🔍 Verificando dependencias...")
    try:
        import pandas as pd
        print("✅ pandas instalado")
    except:
        print("❌ pandas no está instalado. Instalando...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas", "openpyxl"])
    
    try:
        from docx import Document
        print("✅ python-docx instalado")
    except:
        print("❌ python-docx no está instalado. Instalando...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "python-docx"])
    
    # Ejecutar sistema
    main()

🔍 Verificando dependencias...
✅ pandas instalado
✅ python-docx instalado
🤖 SISTEMA AUTOMÁTICO DE EXTRACCIÓN DE AVISOS DE INTERRUPCIÓN
🔧 Inicializando componentes...

🔍 Buscando archivos en: C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\INGRESAR DE FM\IMPUT-AGREGA-ARCHIVOS
📂 Encontrados 30 archivos

[1/30] Procesando: Aviso 06.02.2026 AD-01.docx
--------------------------------------------------------------------------------
  📝 Texto extraído (1017 caracteres)
  📊 Resultados de extracción:
    • Fecha inicio: NO ENCONTRADO
    • Alimentador: AD-01
    • Fecha reposición: NO ENCONTRADO
    • Lugares: Los usuarios conectados al alimentador AD-01, AD-02, AD-04 (SET Andahuasi), parte de las zonas de: S...
    • Confianza: 60.00%
    • Métodos usados: patrones, contexto, ia, inferencia
  ⚠️  Validación: RECHAZADO - Falta fecha de inicio

[2/30] Procesando: Aviso 06.02.2026 CN-02.docx
--------------------------------------------------------------------------------
  📝 Texto extraído (992 

In [10]:
import os
import sys
import re
import pandas as pd
import glob
from datetime import datetime
import json
from collections import defaultdict

# ============================================
# SISTEMA INTELIGENTE DE EXTRACCIÓN
# ============================================
class ExtractorInteligente:
    def __init__(self):
        # Patrones base que se ajustarán automáticamente
        self.patrones = {
            'fecha_hora': self._cargar_patrones_fecha(),
            'alimentador': self._cargar_patrones_alimentador(),
            'lugares': self._cargar_patrones_lugares(),
            'secciones': self._cargar_patrones_secciones()
        }
        self.estadisticas = defaultdict(int)
        self.aprendizaje = self._cargar_aprendizaje()
        
    def _cargar_patrones_fecha(self):
        """Patrones para fechas (se ajustarán automáticamente)"""
        return [
            # Formato dd/mm/yyyy HH:MM
            r'(\d{2}/\d{2}/\d{4}\s+\d{1,2}:\d{2})',
            # Formato dd/mm/yyyy
            r'(\d{2}/\d{2}/\d{4})',
            # Con palabras clave
            r'(?:Fecha|Día|Inicio|Hora)[:\s]*(\d{2}/\d{2}/\d{4}(?:\s+\d{1,2}:\d{2})?)',
            # Diferentes separadores
            r'(\d{2}-\d{2}-\d{4}(?:\s+\d{1,2}:\d{2})?)',
            r'(\d{2}\.\d{2}\.\d{4}(?:\s+\d{1,2}:\d{2})?)'
        ]
    
    def _cargar_patrones_alimentador(self):
        """Patrones para códigos de alimentador"""
        return [
            # AD_01, AD 02, AD_04
            r'([A-Z]{1,3}[-_ ][A-Z0-9]{1,3}(?:\s*,\s*[A-Z]{1,3}[-_ ][A-Z0-9]{1,3})*)',
            # Con palabras clave
            r'(?:Alimentador|Instalación|Circuito|Afectada)[:\s]*([^\n]+)',
            # Solo código
            r'\b([A-Z]{1,3}[-_][A-Z0-9]{1,3})\b'
        ]
    
    def _cargar_patrones_lugares(self):
        """Patrones para lugares afectados"""
        return [
            r'(?:Lugares|Sectores|Clientes|Zonas|Área|Ubicación)[\s\S]{0,50}afectad[ao]s?[:\s\-]*([^\n]+(?:\n[^\n]+){0,10})',
            r'(?:afecta a|ubicados en|sectores)[:\s]*([^\n]+(?:\n[^\n]+){0,10})',
            r'(?:incluye|comprende|abarca)[:\s]*([^\n]+(?:\n[^\n]+){0,10})'
        ]
    
    def _cargar_patrones_secciones(self):
        """Patrones para identificar secciones del documento"""
        return {
            'inicio': ['inicio', 'interrupción', 'ocurrencia', 'día', 'fecha y hora'],
            'alimentador': ['alimentador', 'instalación', 'circuito', 'afectada', 'subestación'],
            'reposicion': ['reposición', 'normalización', 'restablecimiento', 'fin', 'conclusión'],
            'lugares': ['lugares', 'sectores', 'clientes', 'zonas', 'ubicación', 'área', 'afectados']
        }
    
    def _cargar_aprendizaje(self):
        """Carga aprendizaje previo"""
        try:
            with open('aprendizaje_extractor.json', 'r', encoding='utf-8') as f:
                return json.load(f)
        except:
            return {'patrones_exitosos': {}, 'estructuras': []}
    
    def _guardar_aprendizaje(self):
        """Guarda lo aprendido"""
        with open('aprendizaje_extractor.json', 'w', encoding='utf-8') as f:
            json.dump(self.aprendizaje, f, indent=2, ensure_ascii=False)
    
    def leer_documento_word(self, ruta_archivo):
        """Lee cualquier tipo de documento Word"""
        texto = ""
        
        # Método 1: python-docx
        try:
            from docx import Document
            doc = Document(ruta_archivo)
            texto = "\n".join([para.text for para in doc.paragraphs])
            self.estadisticas['metodo_docx'] += 1
            return texto
        except Exception as e1:
            print(f"  Método docx falló: {e1}")
        
        # Método 2: textract (si está disponible)
        try:
            import textract
            texto = textract.process(ruta_archivo).decode('utf-8', errors='ignore')
            self.estadisticas['metodo_textract'] += 1
            return texto
        except Exception as e2:
            print(f"  Método textract falló: {e2}")
        
        # Método 3: antiword (para .doc)
        if ruta_archivo.lower().endswith('.doc'):
            try:
                import subprocess
                result = subprocess.run(['antiword', ruta_archivo], 
                                      capture_output=True, text=True)
                if result.returncode == 0:
                    texto = result.stdout
                    self.estadisticas['metodo_antiword'] += 1
                    return texto
            except Exception as e3:
                print(f"  Método antiword falló: {e3}")
        
        # Método 4: lectura binaria
        try:
            with open(ruta_archivo, 'rb') as f:
                contenido = f.read()
                for encoding in ['utf-8', 'latin-1', 'cp1252', 'iso-8859-1']:
                    try:
                        texto = contenido.decode(encoding, errors='ignore')
                        # Buscar texto que parezca contenido real
                        if len(texto) > 100 and any(x in texto.lower() for x in ['interrupción', 'alimentador', 'afectados']):
                            self.estadisticas['metodo_binario'] += 1
                            return texto
                    except:
                        continue
        except Exception as e4:
            print(f"  Método binario falló: {e4}")
        
        return ""
    
    def analizar_estructura_documento(self, texto):
        """Analiza la estructura del documento para entender su formato"""
        lineas = [linea.strip() for linea in texto.split('\n') if linea.strip()]
        
        estructura = {
            'num_lineas': len(lineas),
            'lineas_con_fecha': [],
            'lineas_con_alimentador': [],
            'lineas_con_lugares': [],
            'patrones_detectados': []
        }
        
        for i, linea in enumerate(lineas):
            # Detectar fechas
            for patron in self.patrones['fecha_hora']:
                if re.search(patron, linea):
                    estructura['lineas_con_fecha'].append({'linea': i, 'texto': linea[:50]})
                    break
            
            # Detectar alimentadores
            for patron in self.patrones['alimentador']:
                if re.search(patron, linea, re.IGNORECASE):
                    estructura['lineas_con_alimentador'].append({'linea': i, 'texto': linea[:50]})
                    break
            
            # Detectar lugares
            if any(palabra in linea.lower() for palabra in ['calle', 'av.', 'jr.', 'urb.', 'sector', 'asoc.', 'distrito']):
                estructura['lineas_con_lugares'].append({'linea': i, 'texto': linea[:100]})
        
        return estructura
    
    def extraer_por_secciones(self, texto, estructura):
        """Extrae datos identificando secciones del documento"""
        datos = {}
        lineas = [linea.strip() for linea in texto.split('\n')]
        
        # Buscar sección de fecha/hora inicio
        for i, linea in enumerate(lineas):
            linea_lower = linea.lower()
            
            # Buscar línea con palabras clave de inicio
            if any(palabra in linea_lower for palabra in self.patrones['secciones']['inicio']):
                # Buscar fecha en esta línea
                for patron in self.patrones['fecha_hora']:
                    match = re.search(patron, linea)
                    if match:
                        datos['fecha_inicio'] = match.group(1).strip()
                        print(f"  ✓ Fecha inicio encontrada en sección 'inicio': {datos['fecha_inicio']}")
                        
                        # Buscar hora si solo hay fecha
                        if ':' not in datos['fecha_inicio'] and i+1 < len(lineas):
                            # Buscar hora en línea siguiente
                            hora_match = re.search(r'(\d{1,2}:\d{2})', lineas[i+1])
                            if hora_match:
                                datos['fecha_inicio'] = f"{datos['fecha_inicio']} {hora_match.group(1)}"
                        
                        break
                if 'fecha_inicio' in datos:
                    break
        
        # Buscar alimentador
        for i, linea in enumerate(lineas):
            linea_lower = linea.lower()
            
            if any(palabra in linea_lower for palabra in self.patrones['secciones']['alimentador']):
                for patron in self.patrones['alimentador']:
                    match = re.search(patron, linea, re.IGNORECASE)
                    if match:
                        alimentador = match.group(1).strip()
                        # Limpiar el alimentador
                        alimentador = re.sub(r'^.*?[:\-]\s*', '', alimentador, flags=re.IGNORECASE)
                        alimentador = re.sub(r'[^\w\s\-,]', '', alimentador)
                        datos['alimentador'] = alimentador
                        print(f"  ✓ Alimentador encontrado: {datos['alimentador']}")
                        break
                if 'alimentador' in datos:
                    break
        
        # Buscar fecha reposición
        for i, linea in enumerate(lineas):
            linea_lower = linea.lower()
            
            if any(palabra in linea_lower for palabra in self.patrones['secciones']['reposicion']):
                for patron in self.patrones['fecha_hora']:
                    match = re.search(patron, linea)
                    if match:
                        datos['fecha_reposicion'] = match.group(1).strip()
                        print(f"  ✓ Fecha reposición encontrada: {datos['fecha_reposicion']}")
                        
                        # Buscar hora si solo hay fecha
                        if ':' not in datos['fecha_reposicion'] and i+1 < len(lineas):
                            hora_match = re.search(r'(\d{1,2}:\d{2})', lineas[i+1])
                            if hora_match:
                                datos['fecha_reposicion'] = f"{datos['fecha_reposicion']} {hora_match.group(1)}"
                        
                        break
                if 'fecha_reposicion' in datos:
                    break
        
        # Buscar lugares
        for i, linea in enumerate(lineas):
            linea_lower = linea.lower()
            
            if any(palabra in linea_lower for palabra in self.patrones['secciones']['lugares']):
                # Esta línea inicia la sección de lugares
                lugares_texto = ""
                
                # Tomar esta línea y las siguientes (hasta 10 líneas o hasta nueva sección)
                for j in range(i, min(i + 15, len(lineas))):
                    # Verificar si es nueva sección
                    if j > i:
                        siguiente_linea_lower = lineas[j].lower()
                        if any(palabra in siguiente_linea_lower for palabra in 
                              ['fecha', 'alimentador', 'reposición', 'nota', 'observación', '--', '---']):
                            break
                    
                    lugares_texto += " " + lineas[j]
                
                # Extraer solo la parte después de las palabras clave
                for patron in self.patrones['lugares']:
                    match = re.search(patron, lugares_texto, re.IGNORECASE | re.DOTALL)
                    if match:
                        lugares = match.group(1).strip()
                        # Limpiar el texto
                        lugares = re.sub(r'\s+', ' ', lugares)
                        lugares = re.sub(r'\n', ', ', lugares)
                        # Eliminar encabezados repetidos
                        lugares = re.sub(r'^(?:Lugares|Sectores|Clientes|Zonas)[\s\S]{0,20}afectad[ao]s?[:\s\-]*', '', lugares, flags=re.IGNORECASE)
                        datos['lugares'] = lugares
                        print(f"  ✓ Lugares encontrados: {datos['lugares'][:100]}...")
                        break
                
                if 'lugares' not in datos and len(lugares_texto) > 20:
                    # Usar el texto completo si no se pudo extraer con patrones
                    datos['lugares'] = lugares_texto.strip()[:500]
                
                break
        
        return datos
    
    def extraer_por_patrones_avanzados(self, texto):
        """Extrae datos usando patrones avanzados y aprendizaje"""
        datos = {}
        
        # ESTRATEGIA 1: Buscar todas las fechas y determinar cuál es inicio y cuál reposición
        todas_fechas = []
        for patron in self.patrones['fecha_hora']:
            matches = re.findall(patron, texto)
            for match in matches:
                if isinstance(match, tuple):
                    match = match[0]  # Tomar el primer grupo si hay grupos
                if match not in todas_fechas:
                    todas_fechas.append(match)
        
        # Ordenar fechas (asumimos que la primera es inicio, la segunda reposición)
        if len(todas_fechas) >= 1:
            datos['fecha_inicio'] = todas_fechas[0]
        
        if len(todas_fechas) >= 2:
            datos['fecha_reposicion'] = todas_fechas[1]
        elif len(todas_fechas) == 1:
            # Si solo hay una fecha, duplicarla para reposición (ajustar después)
            datos['fecha_reposicion'] = todas_fechas[0]
        
        # ESTRATEGIA 2: Buscar alimentador con múltiples patrones
        alimentadores_encontrados = []
        for patron in self.patrones['alimentador']:
            matches = re.findall(patron, texto, re.IGNORECASE)
            for match in matches:
                if isinstance(match, tuple):
                    match = match[0]
                # Limpiar y validar
                match_clean = re.sub(r'[^\w\s\-,]', '', str(match)).strip()
                if len(match_clean) >= 3 and match_clean not in alimentadores_encontrados:
                    alimentadores_encontrados.append(match_clean)
        
        if alimentadores_encontrados:
            datos['alimentador'] = ', '.join(alimentadores_encontrados[:3])  # Máximo 3
        
        # ESTRATEGIA 3: Buscar lugares - método contextual
        # Encontrar la parte del texto que habla de lugares afectados
        palabras_clave_lugares = ['afectados', 'sectores', 'ubicación', 'zona', 'calle', 'av.', 'jr.', 'urb.']
        lineas = texto.split('\n')
        
        secciones_lugares = []
        for i, linea in enumerate(lineas):
            if any(palabra in linea.lower() for palabra in palabras_clave_lugares):
                # Tomar esta línea y contexto
                inicio = max(0, i - 1)
                fin = min(len(lineas), i + 6)  # 5 líneas después
                seccion = ' '.join(lineas[inicio:fin])
                secciones_lugares.append(seccion)
        
        if secciones_lugares:
            # Usar la sección más larga (probablemente la más detallada)
            seccion_lugares = max(secciones_lugares, key=len)
            
            # Limpiar la sección
            lugares = seccion_lugares.strip()
            # Eliminar etiquetas comunes
            lugares = re.sub(r'^(?:Lugares|Sectores|Clientes|Zonas)[\s\S]{0,30}[:\\-]', '', lugares, flags=re.IGNORECASE)
            lugares = re.sub(r'\s+', ' ', lugares)
            
            if len(lugares) > 20:  # Solo si tiene contenido significativo
                datos['lugares'] = lugares[:800]  # Limitar longitud
        
        # Ajustar fechas si es necesario
        if 'fecha_inicio' in datos and 'fecha_reposicion' in datos:
            # Si ambas fechas son iguales (solo fecha), buscar horas diferentes
            if datos['fecha_inicio'] == datos['fecha_reposicion'] and ':' not in datos['fecha_inicio']:
                # Buscar horas en el texto
                horas = re.findall(r'(\d{1,2}:\d{2})', texto)
                if len(horas) >= 2:
                    datos['fecha_inicio'] = f"{datos['fecha_inicio']} {horas[0]}"
                    datos['fecha_reposicion'] = f"{datos['fecha_reposicion']} {horas[1]}"
        
        return datos
    
    def extraer_datos(self, ruta_archivo):
        """Método principal de extracción que combina todas las estrategias"""
        nombre_archivo = os.path.basename(ruta_archivo)
        print(f"\n🔍 Extrayendo datos de: {nombre_archivo}")
        print("-" * 60)
        
        # 1. Leer el documento
        texto = self.leer_documento_word(ruta_archivo)
        
        if not texto or len(texto) < 50:
            print(f"  ❌ Documento vacío o no se pudo leer: {nombre_archivo}")
            return None
        
        print(f"  📝 Texto extraído ({len(texto)} caracteres)")
        
        # 2. Analizar estructura
        estructura = self.analizar_estructura_documento(texto)
        
        # 3. Estrategia 1: Extraer por secciones
        datos_secciones = self.extraer_por_secciones(texto, estructura)
        
        # 4. Estrategia 2: Extraer por patrones avanzados
        datos_patrones = self.extraer_por_patrones_avanzados(texto)
        
        # 5. Combinar resultados (priorizar secciones, luego patrones)
        datos_finales = {}
        
        # Combinar campos, priorizando los que tengan más confianza
        campos = ['fecha_inicio', 'alimentador', 'fecha_reposicion', 'lugares']
        
        for campo in campos:
            # Priorizar datos de secciones (más precisos)
            if campo in datos_secciones and datos_secciones[campo]:
                datos_finales[campo] = datos_secciones[campo]
            elif campo in datos_patrones and datos_patrones[campo]:
                datos_finales[campo] = datos_patrones[campo]
        
        # 6. Validar y formatear
        datos_formateados = self.formatear_datos(datos_finales, nombre_archivo)
        
        # 7. Mostrar resultados
        self.mostrar_resultados_extraction(datos_formateados)
        
        # 8. Aprender de esta extracción
        self.aprender_de_extraccion(texto, datos_formateados, estructura)
        
        return datos_formateados if datos_formateados['completo'] else None
    
    def formatear_datos(self, datos, nombre_archivo):
        """Formatea los datos al formato exacto requerido"""
        formateados = {
            'Fecha y Hora de Inicio de Interrupción': '',
            'Instalación Afectada': '',
            'Fecha y Hora de Reposición': '',
            'Lugares y Sectores Afectados': '',
            'completo': False
        }
        
        # Formatear fecha inicio
        if 'fecha_inicio' in datos:
            fecha = datos['fecha_inicio']
            # Asegurar formato dd/mm/yyyy HH:MM
            fecha = self.normalizar_fecha(fecha)
            formateados['Fecha y Hora de Inicio de Interrupción'] = fecha
        
        # Formatear alimentador
        if 'alimentador' in datos:
            alimentador = datos['alimentador']
            # Limpiar y estandarizar
            alimentador = re.sub(r'\s+', ' ', alimentador).strip()
            alimentador = re.sub(r',\s*,', ',', alimentador)
            # Asegurar que los códigos tengan guión
            alimentador = re.sub(r'([A-Z]{1,3})[\s_]([A-Z0-9]{1,3})', r'\1-\2', alimentador)
            formateados['Instalación Afectada'] = alimentador
        
        # Formatear fecha reposición
        if 'fecha_reposicion' in datos:
            fecha = datos['fecha_reposicion']
            fecha = self.normalizar_fecha(fecha)
            formateados['Fecha y Hora de Reposición'] = fecha
        
        # Formatear lugares
        if 'lugares' in datos:
            lugares = datos['lugares']
            # Limpiar
            lugares = re.sub(r'\s+', ' ', lugares).strip()
            # Poner punto al final
            if lugares and not lugares.endswith('.'):
                lugares += '.'
            # Capitalizar adecuadamente
            lugares = self.capitalizar_lugares(lugares)
            formateados['Lugares y Sectores Afectados'] = lugares
        
        # Verificar si tenemos los datos mínimos
        if (formateados['Fecha y Hora de Inicio de Interrupción'] and 
            formateados['Instalación Afectada']):
            formateados['completo'] = True
        
        return formateados
    
    def normalizar_fecha(self, fecha):
        """Normaliza una fecha al formato dd/mm/yyyy HH:MM"""
        if not fecha:
            return ""
        
        # Limpiar
        fecha = fecha.strip()
        
        # Asegurar formato
        match = re.search(r'(\d{2})[/\-\.](\d{2})[/\-\.](\d{4})', fecha)
        if match:
            dia, mes, anio = match.groups()
            fecha_base = f"{dia}/{mes}/{anio}"
            
            # Buscar hora
            hora_match = re.search(r'(\d{1,2}:\d{2})', fecha)
            if hora_match:
                return f"{fecha_base} {hora_match.group(1)}"
            else:
                return fecha_base
        
        return fecha
    
    def capitalizar_lugares(self, texto):
        """Capitaliza adecuadamente los nombres de lugares"""
        # Palabras que deben mantenerse en mayúsculas
        palabras_mayusculas = ['AV.', 'JR.', 'URB.', 'CALLE', 'AVENIDA', 'ASOC.', 'AA.HH', 
                              'A.H.', 'C.P.', 'KM', 'SECTOR', 'DISTRITO']
        
        # Palabras que deben mantenerse en minúsculas
        palabras_minusculas = ['de', 'del', 'la', 'las', 'el', 'los', 'y', 'e', 'en', 'con']
        
        palabras = texto.split()
        resultado = []
        
        for i, palabra in enumerate(palabras):
            palabra_upper = palabra.upper()
            # Verificar si es una palabra especial
            if any(palabra_upper.startswith(p) for p in palabras_mayusculas):
                resultado.append(palabra_upper)
            elif i > 0 and palabra_upper.lower() in palabras_minusculas:
                resultado.append(palabra_upper.lower())
            else:
                # Capitalizar primera letra
                if len(palabra) > 1:
                    resultado.append(palabra[0].upper() + palabra[1:].lower())
                else:
                    resultado.append(palabra)
        
        return ' '.join(resultado)
    
    def mostrar_resultados_extraction(self, datos):
        """Muestra los resultados de la extracción"""
        print("  📊 Resultados obtenidos:")
        for campo, valor in datos.items():
            if campo != 'completo' and valor:
                print(f"    • {campo}: {valor[:100]}{'...' if len(str(valor)) > 100 else ''}")
        
        if datos['completo']:
            print("  ✅ Extracción COMPLETA")
        else:
            print("  ⚠️  Extracción PARCIAL")
    
    def aprender_de_extraccion(self, texto, datos, estructura):
        """Aprende de esta extracción para mejorar futuras"""
        # Guardar patrones que funcionaron
        if datos['completo']:
            # Extraer patrones específicos que llevaron al éxito
            lineas = texto.split('\n')
            
            for i, linea in enumerate(lineas):
                if datos['Fecha y Hora de Inicio de Interrupción'] in linea:
                    # Guardar contexto de fecha exitosa
                    contexto = ' '.join(lineas[max(0,i-2):min(len(lineas),i+3)])
                    self.aprendizaje.setdefault('contextos_fecha', []).append(contexto[:200])
                
                if datos['Instalación Afectada'].split(',')[0].strip() in linea:
                    contexto = ' '.join(lineas[max(0,i-2):min(len(lineas),i+3)])
                    self.aprendizaje.setdefault('contextos_alimentador', []).append(contexto[:200])
            
            # Guardar estructura exitosa
            self.aprendizaje['estructuras'].append({
                'num_lineas': estructura['num_lineas'],
                'lineas_con_fecha': len(estructura['lineas_con_fecha']),
                'lineas_con_alimentador': len(estructura['lineas_con_alimentador'])
            })
            
            # Limitar a las últimas 100 estructuras
            if len(self.aprendizaje['estructuras']) > 100:
                self.aprendizaje['estructuras'] = self.aprendizaje['estructuras'][-100:]
            
            self._guardar_aprendizaje()

# ============================================
# GENERADOR DE EXCEL INTELIGENTE
# ============================================
class GeneradorExcelInteligente:
    def __init__(self):
        self.extractor = ExtractorInteligente()
        self.datos_procesados = []
    
    def procesar_carpeta(self, ruta_carpeta):
        """Procesa todos los archivos Word en una carpeta"""
        print("=" * 80)
        print("🔍 BUSCANDO ARCHIVOS WORD EN LA CARPETA")
        print("=" * 80)
        
        # Buscar archivos Word
        archivos = []
        for extension in ['*.docx', '*.doc', '*.DOCX', '*.DOC', '*.rtf', '*.txt']:
            archivos.extend(glob.glob(os.path.join(ruta_carpeta, extension)))
        
        if not archivos:
            print("❌ No se encontraron archivos Word en la carpeta.")
            print(f"   Ruta: {ruta_carpeta}")
            print("\n💡 Coloca tus archivos Word en esta carpeta y vuelve a ejecutar.")
            return []
        
        print(f"📁 Encontrados {len(archivos)} archivos para procesar")
        print("-" * 80)
        
        # Procesar cada archivo
        self.datos_procesados = []
        
        for i, archivo in enumerate(archivos, 1):
            print(f"\n[{i}/{len(archivos)}] Procesando archivo...")
            datos = self.extractor.extraer_datos(archivo)
            
            if datos:
                # Agregar información del archivo fuente
                datos['archivo_fuente'] = os.path.basename(archivo)
                self.datos_procesados.append(datos)
                print(f"  ✅ Añadido a la lista de datos")
            else:
                print(f"  ❌ No se pudieron extraer datos válidos")
        
        return self.datos_procesados
    
    def generar_excel(self, ruta_salida, nombre_archivo=None):
        """Genera el archivo Excel con los datos extraídos"""
        if not self.datos_procesados:
            print("❌ No hay datos para generar Excel")
            return None
        
        # Crear DataFrame
        datos_para_excel = []
        
        for datos in self.datos_procesados:
            datos_para_excel.append({
                'Fecha y Hora de Inicio de Interrupción': datos['Fecha y Hora de Inicio de Interrupción'],
                'Instalación Afectada': datos['Instalación Afectada'],
                'Fecha y Hora de Reposición': datos['Fecha y Hora de Reposición'],
                'Lugares y Sectores Afectados': datos['Lugares y Sectores Afectados']
            })
        
        df = pd.DataFrame(datos_para_excel)
        
        # Ordenar por fecha
        try:
            # Crear columna temporal para ordenar
            df['_temp_fecha'] = pd.to_datetime(df['Fecha y Hora de Inicio de Interrupción'], 
                                              dayfirst=True, errors='coerce')
            df = df.sort_values('_temp_fecha')
            df = df.drop('_temp_fecha', axis=1)
        except:
            pass  # Mantener orden original si falla
        
        # Generar nombre de archivo
        if not nombre_archivo:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            nombre_archivo = f"AVISOS_INTERRUPCION_{timestamp}.xlsx"
        
        ruta_completa = os.path.join(ruta_salida, nombre_archivo)
        
        try:
            # Guardar Excel
            df.to_excel(ruta_completa, index=False)
            
            # Guardar también como CSV
            ruta_csv = ruta_completa.replace('.xlsx', '.csv')
            df.to_csv(ruta_csv, index=False, sep=';', encoding='utf-8-sig')
            
            print(f"\n✅ EXCEL GENERADO EXITOSAMENTE: {ruta_completa}")
            print(f"📊 Total de registros: {len(df)}")
            
            # Mostrar contenido
            print("\n" + "=" * 80)
            print("CONTENIDO DEL ARCHIVO GENERADO")
            print("=" * 80)
            
            # Configurar pandas para mostrar todo
            pd.set_option('display.max_columns', None)
            pd.set_option('display.width', None)
            pd.set_option('display.max_colwidth', 100)
            
            print(df.to_string(index=False))
            
            print(f"\n📄 También se creó: {ruta_csv}")
            
            # Mostrar estadísticas del extractor
            self.mostrar_estadisticas()
            
            return ruta_completa
            
        except Exception as e:
            print(f"❌ Error al generar Excel: {e}")
            return None
    
    def mostrar_estadisticas(self):
        """Muestra estadísticas del proceso"""
        print("\n" + "=" * 80)
        print("📈 ESTADÍSTICAS DEL PROCESAMIENTO")
        print("=" * 80)
        
        if hasattr(self.extractor, 'estadisticas'):
            for metodo, count in self.extractor.estadisticas.items():
                print(f"  • {metodo}: {count}")
        
        print(f"  • Archivos procesados: {len(self.datos_procesados)}")
        print(f"  • Métodos de lectura utilizados: {sum(self.extractor.estadisticas.values())}")

# ============================================
# SISTEMA DE PRUEBA Y VALIDACIÓN
# ============================================
class SistemaPruebaAutomatica:
    def __init__(self):
        self.generador = GeneradorExcelInteligente()
    
    def crear_documentos_prueba(self, ruta_carpeta):
        """Crea documentos Word de prueba con diferentes formatos"""
        try:
            from docx import Document
            from docx.shared import Inches
        except:
            print("❌ No se puede crear documentos de prueba. Instala python-docx:")
            print("   pip install python-docx")
            return []
        
        documentos_prueba = [
            {
                'nombre': 'AVISO_AD_01.docx',
                'contenido': [
                    "AVISO DE INTERRUPCIÓN PROGRAMADA",
                    "",
                    "Fecha de inicio: 06/02/2026 14:47",
                    "Alimentador afectado: AD_01, AD 02, AD_04",
                    "Hora de reposición: 06/02/2026 15:07",
                    "",
                    "Lugares afectados:",
                    "Sector Lluvio, Fundo El manatial, Asoc. De viviendas Resindencial.",
                    "",
                    "Motivo: Mantenimiento preventivo",
                    "Duración estimada: 20 minutos"
                ]
            },
            {
                'nombre': 'AVISO_CN-02.docx',
                'contenido': [
                    "INTERRUPCIÓN DE SERVICIO",
                    "",
                    "Día: 06/02/2026",
                    "Hora inicio: 17:10",
                    "Circuito: CN-02",
                    "Normalización: 06/02/2026 20:53",
                    "",
                    "Sectores afectados:",
                    "Urb. Antonia Moreno de Caceres,",
                    "Asoc. Porcicultores.",
                    "",
                    "Observaciones: Trabajos de mejora de red"
                ]
            },
            {
                'nombre': 'AVISO_FA-04.docx',
                'contenido': [
                    "COMUNICADO DE CORTE PROGRAMADO",
                    "",
                    "Instalación afectada: FA-04",
                    "Fecha interrupción: 06/02/2026",
                    "Hora restablecimiento: 22:38",
                    "",
                    "Ubicación del área afectada:",
                    "Av. LSanta Rosa, Asoc. La Florida,",
                    "Jr. Cajamarca, Urb. Los Chasquis.",
                    "",
                    "Contacto: 0800-12345"
                ]
            }
        ]
        
        creados = []
        
        for doc_info in documentos_prueba:
            doc = Document()
            
            for linea in doc_info['contenido']:
                if linea.strip() == "":
                    doc.add_paragraph()  # Párrafo vacío
                else:
                    doc.add_paragraph(linea)
            
            ruta_completa = os.path.join(ruta_carpeta, doc_info['nombre'])
            doc.save(ruta_completa)
            creados.append(ruta_completa)
            print(f"  ✅ Creado: {doc_info['nombre']}")
        
        return creados
    
    def ejecutar_prueba_completa(self, ruta_carpeta):
        """Ejecuta una prueba completa del sistema"""
        print("🧪 INICIANDO PRUEBA AUTOMÁTICA DEL SISTEMA")
        print("=" * 80)
        
        # 1. Crear documentos de prueba
        print("\n1. Creando documentos de prueba...")
        documentos = self.crear_documentos_prueba(ruta_carpeta)
        
        if not documentos:
            print("❌ No se pudieron crear documentos de prueba")
            return False
        
        # 2. Procesar documentos
        print(f"\n2. Procesando {len(documentos)} documentos...")
        datos = self.generador.procesar_carpeta(ruta_carpeta)
        
        if not datos:
            print("❌ No se pudieron extraer datos")
            return False
        
        # 3. Generar Excel
        print("\n3. Generando archivo Excel...")
        excel_generado = self.generador.generar_excel(ruta_carpeta, "PRUEBA_SISTEMA.xlsx")
        
        if excel_generado:
            print(f"\n🎉 PRUEBA EXITOSA: {excel_generado}")
            
            # 4. Verificar contenido
            print("\n4. Verificando contenido generado...")
            try:
                df = pd.read_excel(excel_generado)
                print(f"   • Registros generados: {len(df)}")
                print(f"   • Columnas: {', '.join(df.columns)}")
                
                # Mostrar primeros registros
                print("\n   Primeros registros:")
                print(df.head().to_string(index=False))
                
                return True
            except Exception as e:
                print(f"❌ Error al verificar: {e}")
                return False
        else:
            print("❌ No se pudo generar el Excel")
            return False

# ============================================
# INTERFAZ PRINCIPAL MEJORADA
# ============================================
def main():
    print("=" * 80)
    print("🤖 SISTEMA INTELIGENTE DE EXTRACCIÓN DE AVISOS DE INTERRUPCIÓN")
    print("=" * 80)
    
    # Ruta de trabajo
    ruta_base = r"C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\INGRESAR DE FM"
    ruta_carpeta = os.path.join(ruta_base, "IMPUT-AGREGA-ARCHIVOS")
    
    # Crear carpetas si no existen
    os.makedirs(ruta_carpeta, exist_ok=True)
    
    print(f"📂 Carpeta de trabajo: {ruta_carpeta}")
    print("\n" + "=" * 80)
    
    # Mostrar opciones
    print("OPCIONES DISPONIBLES:")
    print("1. 📁 Procesar archivos Word reales de la carpeta")
    print("2. 🧪 Ejecutar prueba automática del sistema")
    print("3. 📊 Verificar contenido de la carpeta")
    print("4. 🚀 Procesamiento avanzado con aprendizaje")
    
    try:
        opcion = int(input("\nSeleccione una opción (1-4): "))
    except:
        opcion = 1  # Por defecto
    
    sistema = GeneradorExcelInteligente()
    pruebas = SistemaPruebaAutomatica()
    
    if opcion == 1:
        # Procesar archivos reales
        print("\n" + "=" * 80)
        print("🔄 PROCESANDO ARCHIVOS WORD REALES")
        print("=" * 80)
        
        # Verificar qué hay en la carpeta
        archivos = glob.glob(os.path.join(ruta_carpeta, "*"))
        archivos_word = [f for f in archivos if f.lower().endswith(('.docx', '.doc', '.rtf'))]
        
        if not archivos_word:
            print("⚠️  No se encontraron archivos Word en la carpeta.")
            print("\n💡 Sugerencias:")
            print("  1. Coloca tus archivos Word en:")
            print(f"     {ruta_carpeta}")
            print("  2. Los archivos pueden ser .docx, .doc o .rtf")
            print("  3. Ejecuta la opción 2 para una prueba automática")
        else:
            print(f"📁 Encontrados {len(archivos_word)} archivos Word")
            for archivo in archivos_word:
                print(f"  • {os.path.basename(archivo)}")
            
            confirmar = input("\n¿Continuar con el procesamiento? (s/n): ").lower()
            if confirmar == 's':
                datos = sistema.procesar_carpeta(ruta_carpeta)
                if datos:
                    sistema.generar_excel(ruta_carpeta)
    
    elif opcion == 2:
        # Prueba automática
        print("\n" + "=" * 80)
        print("🧪 EJECUTANDO PRUEBA AUTOMÁTICA DEL SISTEMA")
        print("=" * 80)
        
        # Preguntar si limpiar carpeta primero
        contenido = os.listdir(ruta_carpeta)
        if contenido:
            print(f"⚠️  La carpeta contiene {len(contenido)} archivos")
            limpiar = input("¿Limpiar carpeta antes de la prueba? (s/n): ").lower()
            if limpiar == 's':
                for archivo in contenido:
                    try:
                        os.remove(os.path.join(ruta_carpeta, archivo))
                    except:
                        pass
        
        # Ejecutar prueba
        exitosa = pruebas.ejecutar_prueba_completa(ruta_carpeta)
        
        if exitosa:
            print("\n✅ EL SISTEMA FUNCIONA CORRECTAMENTE")
            print("\n💡 Ahora puedes colocar tus archivos Word reales en la carpeta")
            print(f"   y ejecutar la Opción 1: {ruta_carpeta}")
        else:
            print("\n❌ La prueba encontró problemas")
            print("\n🔧 Solución de problemas:")
            print("   1. Verifica que python-docx esté instalado: pip install python-docx")
            print("   2. Asegúrate de tener permisos de escritura en la carpeta")
            print("   3. Intenta ejecutar como administrador si es necesario")
    
    elif opcion == 3:
        # Verificar contenido
        print("\n" + "=" * 80)
        print("📊 CONTENIDO DE LA CARPETA")
        print("=" * 80)
        
        if not os.path.exists(ruta_carpeta):
            print(f"❌ La carpeta no existe: {ruta_carpeta}")
            print("Creando carpeta...")
            os.makedirs(ruta_carpeta, exist_ok=True)
            print(f"✅ Carpeta creada")
        else:
            archivos = os.listdir(ruta_carpeta)
            if not archivos:
                print("📭 La carpeta está vacía")
            else:
                print(f"📁 La carpeta contiene {len(archivos)} archivos:")
                
                # Clasificar archivos
                documentos = []
                otros = []
                
                for archivo in archivos:
                    if archivo.lower().endswith(('.docx', '.doc', '.rtf', '.txt')):
                        documentos.append(archivo)
                    else:
                        otros.append(archivo)
                
                if documentos:
                    print("\n📄 Documentos encontrados:")
                    for doc in documentos:
                        print(f"  • {doc}")
                
                if otros:
                    print("\n📦 Otros archivos:")
                    for otro in otros:
                        print(f"  • {otro}")
        
        print(f"\n📍 Ruta completa: {ruta_carpeta}")
    
    elif opcion == 4:
        # Procesamiento avanzado
        print("\n" + "=" * 80)
        print("🚀 PROCESAMIENTO AVANZADO CON APRENDIZAJE AUTOMÁTICO")
        print("=" * 80)
        
        print("Este modo usa aprendizaje automático para mejorar la extracción.")
        print("Se analizarán los patrones de tus documentos para extraer mejor.")
        
        # Cargar aprendizaje previo
        extractor = ExtractorInteligente()
        if extractor.aprendizaje.get('estructuras'):
            print(f"\n📚 Sistema ya ha aprendido de {len(extractor.aprendizaje['estructuras'])} documentos")
        
        # Procesar
        datos = sistema.procesar_carpeta(ruta_carpeta)
        if datos:
            # Guardar aprendizaje mejorado
            extractor._guardar_aprendizaje()
            print("\n🧠 Aprendizaje guardado para futuras ejecuciones")
            
            # Generar Excel
            sistema.generar_excel(ruta_carpeta, "AVISOS_MEJORADO.xlsx")
    
    else:
        print("❌ Opción no válida. Ejecutando opción 1 por defecto...")
        datos = sistema.procesar_carpeta(ruta_carpeta)
        if datos:
            sistema.generar_excel(ruta_carpeta)
    
    print("\n" + "=" * 80)
    print("🏁 PROCESO COMPLETADO")
    print("=" * 80)
    
    # Mostrar ruta final
    print(f"\n📂 Los archivos generados están en: {ruta_carpeta}")
    
    # Preguntar si abrir carpeta
    if sys.platform == 'win32':
        abrir = input("\n¿Abrir carpeta de resultados? (s/n): ").lower()
        if abrir == 's':
            os.startfile(ruta_carpeta)

# ============================================
# INSTALACIÓN DE DEPENDENCIAS
# ============================================
def instalar_dependencias():
    """Instala todas las dependencias necesarias"""
    print("🔧 Verificando e instalando dependencias...")
    
    dependencias = [
        ('pandas', 'pandas'),
        ('openpyxl', 'openpyxl'),
        ('python-docx', 'python-docx')
    ]
    
    import subprocess
    import importlib
    
    for modulo, paquete in dependencias:
        try:
            if modulo == 'python-docx':
                importlib.import_module('docx')
            else:
                importlib.import_module(modulo)
            print(f"  ✅ {modulo} ya está instalado")
        except ImportError:
            print(f"  📦 Instalando {paquete}...")
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", paquete])
                print(f"  ✅ {paquete} instalado correctamente")
            except Exception as e:
                print(f"  ⚠️  No se pudo instalar {paquete}: {e}")
    
    print("\n✅ Dependencias verificadas")

# ============================================
# EJECUCIÓN PRINCIPAL
# ============================================
if __name__ == "__main__":
    # Instalar dependencias si es necesario
    instalar_dependencias()
    
    # Ejecutar sistema principal
    main()

🔧 Verificando e instalando dependencias...
  ✅ pandas ya está instalado
  ✅ openpyxl ya está instalado
  ✅ python-docx ya está instalado

✅ Dependencias verificadas
🤖 SISTEMA INTELIGENTE DE EXTRACCIÓN DE AVISOS DE INTERRUPCIÓN
📂 Carpeta de trabajo: C:\MICHAEL-CALIDAD-ENERGIA\PROYECTOS\WEB-SCRAPIN\INGRESAR DE FM\IMPUT-AGREGA-ARCHIVOS

OPCIONES DISPONIBLES:
1. 📁 Procesar archivos Word reales de la carpeta
2. 🧪 Ejecutar prueba automática del sistema
3. 📊 Verificar contenido de la carpeta
4. 🚀 Procesamiento avanzado con aprendizaje

🔄 PROCESANDO ARCHIVOS WORD REALES
📁 Encontrados 15 archivos Word
  • Aviso 06.02.2026 AD-01.docx
  • Aviso 06.02.2026 CN-02.docx
  • Aviso 06.02.2026 FA-04.docx
  • Aviso 06.02.2026 HL-09.docx
  • Aviso 06.02.2026 PP-02.docx
  • Aviso 06.02.2026 PP-08.docx
  • Aviso 06.02.2026 SU-53.docx
  • Aviso 06.02.2026 T-08.docx
  • Aviso 06.02.2026 T-23.docx
  • Aviso 06.02.2026 V-05.docx
  • Aviso 07.02.2026 U-12.docx
  • Aviso 07.02.2026 V-13.docx
  • Aviso 08.02.2026 